# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Krishma-adhikari/flyrank-projects/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Krishma-adhikari/flyrank-projects.git

fatal: destination path 'flyrank-projects' already exists and is not an empty directory.


In [2]:
import pandas as pd
df = pd.read_csv('flyrank-projects/data/raw/content_refresh_anonymized.csv')

In [3]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Pages with higher search volume and older content receive higher priority for refresh.

In [5]:
bins = [0,100,500,1000,float("inf")]
labels = ["0-100", "101-500", "501-1000","1000+"]

df["volume_bucket"] = pd.cut(df["search_volume"], bins = bins, labels = labels, include_lowest = True)

In [6]:
bucket_table =(df.groupby("volume_bucket").agg(n = ("search_volume","size"),avg_impressions = ("impressions_90d","mean")).reset_index())

/tmp/ipykernel_21572/2308866680.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table =(df.groupby("volume_bucket").agg(n = ("search_volume","size"),avg_impressions = ("impressions_90d","mean")).reset_index())


In [7]:
bucket_table

,volume_bucket,n,avg_impressions
0,0-100,24483,5612.896745
1,101-500,2021,5599.972786
2,501-1000,468,5135.705128
3,1000+,560,6486.880357


Verdict = Mixed


Reason-> Higher search volume generally leads to more impressions, but the trend is not consistent across all buckets.

In [8]:
bins = [0,10,20,50,100,500]
labels = ["0-10", "10-20", "20-50","50-100","100-500"]

df["days_bucket"] = pd.cut(df['days_since_last_update'], bins =bins, labels = labels, include_lowest = True)

In [9]:
bucket2_table =(df.groupby("days_bucket").agg(n = ("days_since_last_update","size"),avg_clicks = ("clicks_90d","mean")).reset_index())

/tmp/ipykernel_21572/216428559.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2_table =(df.groupby("days_bucket").agg(n = ("days_since_last_update","size"),avg_clicks = ("clicks_90d","mean")).reset_index())


In [10]:
bucket2_table

,days_bucket,n,avg_clicks
0,0-10,2667,6.448819
1,10-20,13199,15.006819
2,20-50,4736,14.250633
3,50-100,208,0.437500
4,100-500,9190,21.769750


Verdict -> Mixed

Reason :
Pages that have not been updated for longer periods generally receive more clicks, although the relationship is inconsistent across buckets.


In [11]:
# Rule:
def score_rule(row):
  score = 0
  if row["search_volume"] > 100:
    score+= 60
  if row["days_since_last_update"] > 180:
    score += 40
  return score

In [12]:
df["score"] = df.apply(score_rule, axis = 1)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# rule for reason code
def reason_code(row):
  if row["score"]> 0:
    return "STALE_HIGH_VOLUME"
  return ""

In [14]:
df["reason_code"] = df.apply(reason_code, axis = 1)

In [15]:
print(df["search_volume"].max())
print(df["days_since_last_update"].describe())

74000.0
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


In [16]:
# rule for action label
def action_label(row):
  if row["score"]>0:
    return "Refresh Content"
  return ""

In [17]:
df["action_label"] = df.apply(action_label, axis = 1)

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

The reason code represents the baseline rule and does not distinguish whether one or both signals contributed to the score.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = df.sort_values("score",ascending = False)

In [19]:
df.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,volume_bucket,days_bucket,score,reason_code,action_label
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.0,low,page_1,down,-95.6,101-500,100-500,100,STALE_HIGH_VOLUME,Refresh Content
15947,content_40e140ba2934,client_8722616204,720.0,0.29,LOW,0.78,keyword article,transactional,3306.0,21228.0,...,0.0,low,page_1,flat,NaN,501-1000,100-500,100,STALE_HIGH_VOLUME,Refresh Content
23619,content_24abafed9707,client_8722616204,480.0,0.00,LOW,0.00,keyword article,transactional,3246.0,21393.0,...,0.0,low,top_3,down,-100.0,101-500,100-500,100,STALE_HIGH_VOLUME,Refresh Content
1659,content_bbca724138f2,client_6208ef0f77,1600.0,0.43,MEDIUM,3.76,keyword article,transactional,5614.0,37325.0,...,0.0,low,striking,down,-100.0,1000+,100-500,100,STALE_HIGH_VOLUME,Refresh Content
12008,content_b1bc831deff1,client_4e07408562,4400.0,0.43,MEDIUM,0.09,keyword article,informational,2958.0,17932.0,...,0.0,good,striking,down,-93.4,1000+,20-50,60,STALE_HIGH_VOLUME,Refresh Content
11985,content_24a63ec7c93b,client_4e07408562,3600.0,0.02,LOW,0.59,keyword article,informational,2949.0,17935.0,...,0.0,moderate,striking,stable,-9.4,1000+,0-10,60,STALE_HIGH_VOLUME,Refresh Content
11999,content_3737573e6c27,client_19581e27de,210.0,0.02,LOW,0.08,keyword article,transactional,NaN,NaN,...,0.0,good,striking,down,-35.2,101-500,20-50,60,STALE_HIGH_VOLUME,Refresh Content
11969,content_abfd15f46882,client_4e07408562,140.0,0.78,HIGH,6.21,keyword article,informational,2823.0,18469.0,...,0.0,good,striking,stable,-12.7,101-500,20-50,60,STALE_HIGH_VOLUME,Refresh Content
11937,content_39c1b1ccc234,client_19581e27de,110.0,0.35,MEDIUM,0.18,keyword article,transactional,NaN,NaN,...,0.0,moderate,page_1,stable,-0.3,101-500,100-500,60,STALE_HIGH_VOLUME,Refresh Content
11951,content_30597609fcc6,client_e629fa6598,260.0,0.01,LOW,0.02,keyword article,informational,NaN,NaN,...,0.0,moderate,striking,up,37.2,101-500,20-50,60,STALE_HIGH_VOLUME,Refresh Content


In [20]:
top10 = df.head(10)[[
    "content_id",
    "score",
    "reason_code",
    "action_label"
]]

top10

,content_id,score,reason_code,action_label
21984,content_02b0d6e30129,100,STALE_HIGH_VOLUME,Refresh Content
15947,content_40e140ba2934,100,STALE_HIGH_VOLUME,Refresh Content
23619,content_24abafed9707,100,STALE_HIGH_VOLUME,Refresh Content
1659,content_bbca724138f2,100,STALE_HIGH_VOLUME,Refresh Content
12008,content_b1bc831deff1,60,STALE_HIGH_VOLUME,Refresh Content
11985,content_24a63ec7c93b,60,STALE_HIGH_VOLUME,Refresh Content
11999,content_3737573e6c27,60,STALE_HIGH_VOLUME,Refresh Content
11969,content_abfd15f46882,60,STALE_HIGH_VOLUME,Refresh Content
11937,content_39c1b1ccc234,60,STALE_HIGH_VOLUME,Refresh Content
11951,content_30597609fcc6,60,STALE_HIGH_VOLUME,Refresh Content


## Top-10 Review

1. **Action:** Refresh Content  
   **Reason:** The page received a high score because it matched the baseline rule using search volume and/or days since the last update.  
   **Confidence:** High.  
   **What would make it wrong?** The page may already have a strong CTR or good ranking.

2. **Action:** Refresh Content  
   **Reason:** The page met the baseline scoring criteria.  
   **Confidence:** High.  
   **What would make it wrong?** The content may already be performing well despite being old.

3. **Action:** Refresh Content  
   **Reason:** High search volume and/or stale content increased the priority score.  
   **Confidence:** Medium.  
   **What would make it wrong?** The rule does not consider CTR or average position.

4. **Action:** Refresh Content  
   **Reason:** The page matched the selected baseline signals.  
   **Confidence:** Medium.  
   **What would make it wrong?** The page may have been updated recently or already have good engagement.

5. **Action:** Refresh Content  
   **Reason:** The page received a high baseline score.  
   **Confidence:** Medium.  
   **What would make it wrong?** Content quality or technical SEO issues may be more important than freshness.

6. **Action:** Refresh Content  
   **Reason:** The baseline rule identified this page as a refresh candidate.  
   **Confidence:** High.  
   **What would make it wrong?** Strong existing rankings could make refreshing unnecessary.

7. **Action:** Refresh Content  
   **Reason:** High search volume and/or stale content matched the rule.  
   **Confidence:** Medium.  
   **What would make it wrong?** Another SEO signal not included in the rule may explain the page's performance.

8. **Action:** Refresh Content  
   **Reason:** The page was prioritized by the baseline scoring rule.  
   **Confidence:** Medium.  
   **What would make it wrong?** The page may already satisfy user intent without needing updates.

9. **Action:** Refresh Content  
   **Reason:** The page received a high action score.  
   **Confidence:** Medium.  
   **What would make it wrong?** Engagement and ranking metrics may suggest no refresh is needed.

10. **Action:** Refresh Content  
    **Reason:** The page met the baseline rule conditions.  
    **Confidence:** High.  
    **What would make it wrong?** The recommendation is based on only two signals and ignores other important SEO metrics.

Confidence is higher when both search volume and stale content contribute to the score (100), and lower when only one rule contributes (60 or 40).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-scoring pages may already have good rankings or strong CTR, so refreshing them may not provide much benefit. My rule only uses search volume and days since last update, so it ignores other useful SEO signals such as CTR, average position, and engagement.

The baseline rule only uses search_volume and days_since_last_update, which are available before making a refresh decision. I did not use FlyRank product flags, target labels, or future-window metrics such as clicks_last_30d or impressions_last_30d when calculating the score, so I did not observe obvious data leakage.

In [21]:
import os
os.makedirs("work/outputs", exist_ok = True)
df.to_csv("work/outputs/baseline_action_score.csv",index = False)

## Self-check

Before you submit, confirm each line honestly:

- [Y] Every section above is filled — markdown thinking AND the code that backs it
- [Y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Y] No client names, URLs, or private queries anywhere
- [Y] My claims use careful words: observed, measured, directional, decision-support
- [Y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.